# Deep dive 9/13: `call_transcripts`

Dictionary declares 200,000 rows, Source: Contact Center, Partition: daily. Every column, Bronze and Silver, per your standing instruction.

`table_specs.py`'s comment above this spec flags two known, intentional non-fixes worth directly confirming rather than trusting from memory:

1. **`full_text`/`customer_text`/`agent_text` carry unrendered template placeholders** (`'{monto}'`, `'{moneda}'`, `'{limite}'`) in the raw data -- kept as-is on purpose (guessing real values would fabricate data). Section 11 quantifies how many rows actually show this.
2. **`mentioned_entities` is a JSON-shaped string** (e.g. `'{"account_numbers": 0, "dates": 1, ...}'`), kept as VARCHAR rather than parsed. Section 12 spot-checks it's still consistently JSON-shaped.

Dictionary NOT NULL: `transcript_id`, `interaction_id`, `process_date`, `customer_id`, `agent_id`, `full_text`, `detected_language`, `transcription_model`, `duration_seconds` -- **9 columns**. `contracts.py`'s current `required` tuple only has 7 of these -- **`transcription_model` and `duration_seconds` are both missing**, checked in section 3 as fix candidates. Nullable: `customer_text`, `agent_text`, `detected_accent`, `accent_confidence`, `detected_keywords`, `mentioned_entities`, `detected_intents`, `main_topics`, `audio_quality`.

Also new this table: since `call_center_interactions` (8/13) is already checked, section 7 cross-validates `call_transcripts.customer_id`/`agent_id` against the *parent interaction's own* `customer_id`/`agent_id` for the same `interaction_id` -- they should always agree.

Run top to bottom. Checklist near the end; free cells after that.

In [1]:
import duckdb
import pandas as pd

pd.set_option("display.max_rows", 1000)
pd.set_option("display.min_rows", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 100)

con = duckdb.connect("data/latam_bank.duckdb")

def q(sql, params=None):
    """Run a query and return a pandas DataFrame."""
    return con.execute(sql, params or []).df()

def q_full(sql, params=None):
    """Like q(), but shows every cell's full text with no truncation."""
    with pd.option_context("display.max_colwidth", None):
        return q(sql, params)

## 0. Sample rows -- Bronze and Silver, excluding the long text fields (checked separately below)

In [2]:
sample_ids = q("SELECT transcript_id FROM bronze.call_transcripts ORDER BY transcript_id LIMIT 20")["transcript_id"].tolist()
ids_sql = ", ".join(f"'{i}'" for i in sample_ids)

cols = "transcript_id, interaction_id, process_date, customer_id, agent_id, detected_language, detected_accent, accent_confidence, transcription_model, audio_quality, duration_seconds"
print("Bronze:")
display(q(f"SELECT {cols} FROM bronze.call_transcripts WHERE transcript_id IN ({ids_sql}) ORDER BY transcript_id"))
print("Silver:")
display(q(f"SELECT {cols} FROM silver.fact_call_transcripts WHERE transcript_id IN ({ids_sql}) ORDER BY transcript_id"))

Bronze:


,transcript_id,interaction_id,process_date,customer_id,agent_id,detected_language,detected_accent,accent_confidence,transcription_model,audio_quality,duration_seconds
0,TRS-0008H44LYDZTV83G1X2Y,INT-ZAE8GNELVTD28N6M,2024-09-17,CLI-NJIW9TUXZP1W,AGT-ZKZMQJ4VSL,es,argentine,0.83,Whisper v3,High,444.0
1,TRS-000GALV0HC33GNVERXH4,INT-CPJGMJMWYQSZJYI0,2023-07-29,CLI-Q2RZOP2J0EJR,AGT-3HALJKERVV,es,mexican,0.97,AWS Transcribe,High,NaN
2,TRS-00122PA6FRZ77R95GQVT,INT-2HDC9HJRZVIXXUKD,2023-11-21,CLI-AFI2SD5L83P7,AGT-ERBZDOFKYX,es,argentine,0.83,AWS Transcribe,High,638.0
3,TRS-0017VTLTUMT7W1YKU204,INT-VWX2ZTPC0CA7QV57,2024-11-26,CLI-RNGTIO00OKH6,AGT-O8NXZF1HJ6,es,NaN,0.77,Google STT,High,228.0
4,TRS-00190TX8OYC1W7TPAOUH,INT-89PXG7BEHO6PLZD9,2025-04-29,CLI-A1OC12R6AZGE,AGT-GRF6UL86F3,es,NaN,0.76,Whisper v3,High,57.0
5,TRS-001BQYA984UMD19HFQCI,INT-VEBKIYB0SXAYOLOP,2023-11-14,CLI-6W5577LYK46O,AGT-RLIC03EY4X,es,NaN,0.92,Google STT,Medium,221.0
6,TRS-001SJ3EZ15W8B5FPJGEF,INT-4I7NFS3HTYRT3NYW,2023-10-12,CLI-EBPXT0HY6WTY,AGT-9U9K7MKVAX,es,colombian,0.88,Whisper v3,Medium,187.0
7,TRS-001XJN0W58CPAMISVGY0,INT-BZX5FT23S060MRER,2025-11-15,CLI-L9LWC71GL67Y,AGT-FL833OQDNH,es,NaN,0.99,Azure Speech,NaN,147.0
8,TRS-002R3I26091X563XM4VY,INT-KU3HQKRBJ4Y3IJ2U,2024-05-05,CLI-TF5BXTYWYLVI,AGT-36I8H8V3IO,es,mexican,0.89,Azure Speech,High,130.0
9,TRS-003NQSE5CULQQ0ZFFH92,INT-V1ST146TGJMKXA79,2024-03-04,CLI-6KNC29BTWLD5,AGT-ZGQ4FYLRK2,es,NaN,0.76,Azure Speech,High,174.0


Silver:


,transcript_id,interaction_id,process_date,customer_id,agent_id,detected_language,detected_accent,accent_confidence,transcription_model,audio_quality,duration_seconds
0,TRS-0008H44LYDZTV83G1X2Y,INT-ZAE8GNELVTD28N6M,2024-09-17,CLI-NJIW9TUXZP1W,AGT-ZKZMQJ4VSL,es,argentine,0.83,Whisper v3,High,444.0
1,TRS-000GALV0HC33GNVERXH4,INT-CPJGMJMWYQSZJYI0,2023-07-29,CLI-Q2RZOP2J0EJR,AGT-3HALJKERVV,es,mexican,0.97,AWS Transcribe,High,NaN
2,TRS-00122PA6FRZ77R95GQVT,INT-2HDC9HJRZVIXXUKD,2023-11-21,CLI-AFI2SD5L83P7,AGT-ERBZDOFKYX,es,argentine,0.83,AWS Transcribe,High,638.0
3,TRS-0017VTLTUMT7W1YKU204,INT-VWX2ZTPC0CA7QV57,2024-11-26,CLI-RNGTIO00OKH6,AGT-O8NXZF1HJ6,es,NaN,0.77,Google STT,High,228.0
4,TRS-00190TX8OYC1W7TPAOUH,INT-89PXG7BEHO6PLZD9,2025-04-29,CLI-A1OC12R6AZGE,AGT-GRF6UL86F3,es,NaN,0.76,Whisper v3,High,57.0
5,TRS-001BQYA984UMD19HFQCI,INT-VEBKIYB0SXAYOLOP,2023-11-14,CLI-6W5577LYK46O,AGT-RLIC03EY4X,es,NaN,0.92,Google STT,Medium,221.0
6,TRS-001SJ3EZ15W8B5FPJGEF,INT-4I7NFS3HTYRT3NYW,2023-10-12,CLI-EBPXT0HY6WTY,AGT-9U9K7MKVAX,es,colombian,0.88,Whisper v3,Medium,187.0
7,TRS-001XJN0W58CPAMISVGY0,INT-BZX5FT23S060MRER,2025-11-15,CLI-L9LWC71GL67Y,AGT-FL833OQDNH,es,NaN,0.99,Azure Speech,NaN,147.0
8,TRS-002R3I26091X563XM4VY,INT-KU3HQKRBJ4Y3IJ2U,2024-05-05,CLI-TF5BXTYWYLVI,AGT-36I8H8V3IO,es,mexican,0.89,Azure Speech,High,130.0
9,TRS-003NQSE5CULQQ0ZFFH92,INT-V1ST146TGJMKXA79,2024-03-04,CLI-6KNC29BTWLD5,AGT-ZGQ4FYLRK2,es,NaN,0.76,Azure Speech,High,174.0


## 1. Column inventory -- names AND types, Bronze vs. Silver

In [3]:
expected_silver_columns = [
    "transcript_id", "interaction_id", "process_date", "customer_id", "agent_id", "full_text",
    "customer_text", "agent_text", "detected_language", "detected_accent", "accent_confidence",
    "detected_keywords", "mentioned_entities", "detected_intents", "main_topics",
    "transcription_model", "audio_quality", "duration_seconds",
]

actual_silver_columns = q("""
    SELECT column_name FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='fact_call_transcripts'
    ORDER BY ordinal_position
""")["column_name"].tolist()

missing = [c for c in expected_silver_columns if c not in actual_silver_columns]
extra = [c for c in actual_silver_columns if c not in expected_silver_columns]
print("Missing from silver.fact_call_transcripts:", missing or "none")
print("Extra in silver.fact_call_transcripts:", extra or "none")
print()
print("Column order as built:", actual_silver_columns)

Missing from silver.fact_call_transcripts: none
Extra in silver.fact_call_transcripts: none

Column order as built: ['transcript_id', 'interaction_id', 'process_date', 'customer_id', 'agent_id', 'full_text', 'customer_text', 'agent_text', 'detected_language', 'detected_accent', 'accent_confidence', 'detected_keywords', 'mentioned_entities', 'detected_intents', 'main_topics', 'transcription_model', 'audio_quality', 'duration_seconds']


In [4]:
print("Bronze columns + types:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='bronze' AND table_name='call_transcripts' ORDER BY ordinal_position
"""))

print("Silver columns + types:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='fact_call_transcripts' ORDER BY ordinal_position
"""))

Bronze columns + types:


,column_name,data_type
0,transcript_id,VARCHAR
1,interaction_id,VARCHAR
2,process_date,VARCHAR
3,customer_id,VARCHAR
4,agent_id,VARCHAR
5,full_text,VARCHAR
6,customer_text,VARCHAR
7,agent_text,VARCHAR
8,detected_language,VARCHAR
9,detected_accent,VARCHAR


Silver columns + types:


,column_name,data_type
0,transcript_id,VARCHAR
1,interaction_id,VARCHAR
2,process_date,DATE
3,customer_id,VARCHAR
4,agent_id,VARCHAR
5,full_text,VARCHAR
6,customer_text,VARCHAR
7,agent_text,VARCHAR
8,detected_language,VARCHAR
9,detected_accent,VARCHAR


## 2. Row-count reconciliation + dedup

In [5]:
bronze_total = q("SELECT count(*) AS n FROM bronze.call_transcripts")["n"][0]
bronze_distinct_pk = q("SELECT count(DISTINCT transcript_id) AS n FROM bronze.call_transcripts")["n"][0]
silver_total = q("SELECT count(*) AS n FROM silver.fact_call_transcripts")["n"][0]
silver_distinct_pk = q("SELECT count(DISTINCT transcript_id) AS n FROM silver.fact_call_transcripts")["n"][0]

print(f"bronze.call_transcripts:      {bronze_total:,} rows, {bronze_distinct_pk:,} distinct transcript_id")
print(f"silver.fact_call_transcripts: {silver_total:,} rows, {silver_distinct_pk:,} distinct transcript_id")
print()
print("Dictionary declares 200,000 rows -- matches" if bronze_total == 200_000 else f"Dictionary declares 200,000 rows -- MISMATCH, Bronze has {bronze_total:,}")
print("Duplicate raw rows in Bronze:", bronze_total - bronze_distinct_pk)
print("OK -- Silver row count matches Bronze distinct PK count" if silver_total == bronze_distinct_pk
      else "MISMATCH -- investigate")

bronze.call_transcripts:      171,321 rows, 171,321 distinct transcript_id
silver.fact_call_transcripts: 171,321 rows, 171,321 distinct transcript_id

Dictionary declares 200,000 rows -- MISMATCH, Bronze has 171,321
Duplicate raw rows in Bronze: 0
OK -- Silver row count matches Bronze distinct PK count


## 3. NOT NULL audit -- Bronze raw vs. Silver typed, all 9 dictionary-declared columns

Including `transcription_model` and `duration_seconds`, both missing from `contracts.py`'s `required` tuple -- checked here as fix candidates.

In [6]:
declared_not_null = [
    "transcript_id", "interaction_id", "process_date", "customer_id", "agent_id",
    "full_text", "detected_language", "transcription_model", "duration_seconds",
]
currently_required = {"transcript_id","interaction_id","process_date","customer_id","agent_id","full_text","detected_language"}

rows = []
for col in declared_not_null:
    bronze_nulls = q(f"""
        SELECT count(*) AS n FROM bronze.call_transcripts
        WHERE {col} IS NULL OR trim(CAST({col} AS VARCHAR)) = ''
    """)["n"][0]
    silver_nulls = q(f"""
        SELECT count(*) AS n FROM silver.fact_call_transcripts WHERE {col} IS NULL
    """)["n"][0]
    rows.append({"column": col, "declared": "NOT NULL", "in_contracts_required": col in currently_required,
                 "bronze_nulls": bronze_nulls, "silver_nulls": silver_nulls,
                 "cast_introduced_nulls": silver_nulls - bronze_nulls})

pd.DataFrame(rows)

,column,declared,in_contracts_required,bronze_nulls,silver_nulls,cast_introduced_nulls
0,transcript_id,NOT NULL,True,0,0,0
1,interaction_id,NOT NULL,True,0,0,0
2,process_date,NOT NULL,True,0,0,0
3,customer_id,NOT NULL,True,0,0,0
4,agent_id,NOT NULL,True,0,0,0
5,full_text,NOT NULL,True,0,0,0
6,detected_language,NOT NULL,True,0,0,0
7,transcription_model,NOT NULL,False,0,0,0
8,duration_seconds,NOT NULL,False,24029,24029,0


## 4. Nullable columns -- population rate, Bronze vs. Silver

In [7]:
nullable_cols = ["customer_text", "agent_text", "detected_accent", "accent_confidence",
                  "detected_keywords", "mentioned_entities", "detected_intents", "main_topics",
                  "audio_quality"]
rows = []
total = q("SELECT count(*) AS n FROM bronze.call_transcripts")["n"][0]
for col in nullable_cols:
    bronze_populated = q(f"""
        SELECT count(*) AS n FROM bronze.call_transcripts
        WHERE {col} IS NOT NULL AND trim(CAST({col} AS VARCHAR)) <> ''
    """)["n"][0]
    silver_populated = q(f"""
        SELECT count(*) AS n FROM silver.fact_call_transcripts WHERE {col} IS NOT NULL
    """)["n"][0]
    rows.append({"column": col, "total": total,
                 "bronze_populated": bronze_populated, "bronze_pct": round(100*bronze_populated/total, 2),
                 "silver_populated": silver_populated, "silver_pct": round(100*silver_populated/total, 2)})

pd.DataFrame(rows)

,column,total,bronze_populated,bronze_pct,silver_populated,silver_pct
0,customer_text,171321,171321,100.00,171321,100.00
1,agent_text,171321,171321,100.00,171321,100.00
2,detected_accent,171321,108238,63.18,108238,63.18
3,accent_confidence,171321,154180,89.99,154180,89.99
4,detected_keywords,171321,162524,94.87,162524,94.87
5,mentioned_entities,171321,154157,89.98,154157,89.98
6,detected_intents,171321,162864,95.06,162864,95.06
7,main_topics,171321,171321,100.00,171321,100.00
8,audio_quality,171321,162683,94.96,162683,94.96


## 5. Domain checks -- `detected_language`, `transcription_model`, `audio_quality`

None of these three are currently gated in `contracts.py`. Dictionary doesn't give an explicit value list for `detected_language`, and only gives examples for `transcription_model` ("Whisper, Google STT, etc.") -- discovering the real domains directly.

In [8]:
print("detected_language:")
display(q("SELECT detected_language, count(*) AS n FROM bronze.call_transcripts GROUP BY 1 ORDER BY n DESC"))
print("transcription_model:")
display(q("SELECT transcription_model, count(*) AS n FROM bronze.call_transcripts GROUP BY 1 ORDER BY n DESC"))
print("audio_quality -- dictionary: High, Medium, Low")
display(q("SELECT audio_quality, count(*) AS n FROM bronze.call_transcripts GROUP BY 1 ORDER BY n DESC"))

detected_language:


,detected_language,n
0,es,171321


transcription_model:


,transcription_model,n
0,AWS Transcribe,43117
1,Whisper v3,42803
2,Google STT,42740
3,Azure Speech,42661


audio_quality -- dictionary: High, Medium, Low


,audio_quality,n
0,High,114371
1,Medium,40350
2,NaN,8638
3,Low,7962


## 6. FK integrity -- `interaction_id`, `customer_id`, `agent_id`, reproduced directly

In [9]:
fk_checks = [
    ("interaction_id", "silver.fact_call_center_interactions", "interaction_id"),
    ("customer_id", "silver.dim_customers", "customer_id"),
    ("agent_id", "silver.dim_service_agents", "agent_id"),
]
rows = []
for child_col, parent_table, parent_col in fk_checks:
    total = q(f"SELECT count(*) AS n FROM silver.fact_call_transcripts WHERE {child_col} IS NOT NULL")["n"][0]
    orphans = q(f"""
        SELECT count(*) AS n FROM silver.fact_call_transcripts t
        WHERE t.{child_col} IS NOT NULL
          AND NOT EXISTS (SELECT 1 FROM {parent_table} p WHERE p.{parent_col} = t.{child_col})
    """)["n"][0]
    rows.append({"fk": f"{child_col} -> {parent_table}.{parent_col}", "populated": total,
                 "orphans": orphans, "orphan_pct": round(100*orphans/total, 4) if total else None})

pd.DataFrame(rows)

,fk,populated,orphans,orphan_pct
0,interaction_id -> silver.fact_call_center_interactions.interaction_id,171321,0,0.0
1,customer_id -> silver.dim_customers.customer_id,171321,0,0.0
2,agent_id -> silver.dim_service_agents.agent_id,171321,0,0.0


## 7. Cross-table consistency -- does a transcript's `customer_id`/`agent_id` always match its parent interaction's own?

Every `call_transcripts` row points at a `call_center_interactions` row via `interaction_id`, and both tables independently carry `customer_id`/`agent_id`. They should always agree -- a transcript for interaction X naming a different customer than interaction X itself would be a real data-integrity problem, not just a missing FK.

In [10]:
q("""
    SELECT
        count(*) AS total,
        count(*) FILTER (WHERE t.customer_id = i.customer_id) AS customer_id_matches,
        count(*) FILTER (WHERE t.customer_id <> i.customer_id) AS customer_id_mismatches,
        count(*) FILTER (WHERE t.agent_id = i.agent_id) AS agent_id_matches,
        count(*) FILTER (WHERE t.agent_id <> i.agent_id) AS agent_id_mismatches
    FROM silver.fact_call_transcripts t
    JOIN silver.fact_call_center_interactions i ON i.interaction_id = t.interaction_id
""")

,total,customer_id_matches,customer_id_mismatches,agent_id_matches,agent_id_mismatches
0,171321,171321,0,171321,0


## 8. `accent_confidence` -- range sanity (dictionary: 0-1)

In [11]:
q("""
    SELECT
        typeof(accent_confidence) AS silver_type,
        min(accent_confidence) AS min_conf, max(accent_confidence) AS max_conf,
        count(*) FILTER (WHERE accent_confidence < 0 OR accent_confidence > 1) AS out_of_range,
        count(*) FILTER (WHERE accent_confidence IS NOT NULL) AS populated, count(*) AS total
    FROM silver.fact_call_transcripts
    GROUP BY ALL
""")

,silver_type,min_conf,max_conf,out_of_range,populated,total
0,DOUBLE,0.75,0.99,0,154180,171321


## 9. `duration_seconds` -- range sanity + cross-check against the parent interaction's own `duration_seconds`

Both tables have a `duration_seconds` column for what should be the same underlying call -- checking whether they actually agree.

In [12]:
q("""
    SELECT
        typeof(duration_seconds) AS silver_type,
        min(duration_seconds) AS min_dur, max(duration_seconds) AS max_dur,
        count(*) FILTER (WHERE duration_seconds < 0) AS negative,
        count(*) FILTER (WHERE duration_seconds IS NOT NULL) AS populated, count(*) AS total
    FROM silver.fact_call_transcripts
    GROUP BY ALL
""")

,silver_type,min_dur,max_dur,negative,populated,total
0,DOUBLE,30.0,1151.0,0,147292,171321


In [13]:
q("""
    SELECT
        count(*) AS total,
        count(*) FILTER (WHERE t.duration_seconds = i.duration_seconds) AS exact_match,
        count(*) FILTER (WHERE t.duration_seconds <> i.duration_seconds) AS differs,
        round(avg(abs(t.duration_seconds - i.duration_seconds)), 2) AS avg_abs_diff_seconds
    FROM silver.fact_call_transcripts t
    JOIN silver.fact_call_center_interactions i ON i.interaction_id = t.interaction_id
    WHERE t.duration_seconds IS NOT NULL AND i.duration_seconds IS NOT NULL
""")

,total,exact_match,differs,avg_abs_diff_seconds
0,147292,147292,0,0.0


## 10. `detected_accent` -- domain check

In [14]:
q("SELECT detected_accent, count(*) AS n FROM bronze.call_transcripts GROUP BY 1 ORDER BY n DESC")

,detected_accent,n
0,NaN,63083
1,mexican,54152
2,colombian,32284
3,argentine,21802


## 11. Template placeholder check in `full_text`/`customer_text`/`agent_text`

`table_specs.py` flags this as a known, intentional non-fix -- confirming how widespread it actually is.

In [15]:
total = q("SELECT count(*) AS n FROM bronze.call_transcripts")["n"][0]
for col in ["full_text", "customer_text", "agent_text"]:
    n = q(f"""
        SELECT count(*) AS n FROM bronze.call_transcripts
        WHERE {col} LIKE '%{{monto}}%' OR {col} LIKE '%{{moneda}}%' OR {col} LIKE '%{{limite}}%'
    """)["n"][0]
    print(f"{col}: {n:,} / {total:,} rows contain an unrendered template placeholder ({100*n/total:.2f}%)")

full_text: 171,321 / 171,321 rows contain an unrendered template placeholder (100.00%)
customer_text: 0 / 171,321 rows contain an unrendered template placeholder (0.00%)
agent_text: 171,321 / 171,321 rows contain an unrendered template placeholder (100.00%)


In [16]:
q_full("""
    SELECT transcript_id, full_text FROM bronze.call_transcripts
    WHERE full_text LIKE '%{monto}%' OR full_text LIKE '%{moneda}%' OR full_text LIKE '%{limite}%'
    LIMIT 5
""")

,transcript_id,full_text
0,TRS-PPFGAVR5FA7R3DCC91G4,"Cliente: Hola, buenos días. Quisiera saber cuál es mi saldo actual en mi cuenta de ahorros.\n\nA..."
1,TRS-DD79XNL6Q5DZNKOUR81B,"Cliente: Hola, buenos días. Quisiera saber cuál es mi saldo actual en mi cuenta de ahorros.\n\nA..."
2,TRS-E7ZI313MSF2DZJYU0POJ,"Cliente: Buenas tardes, necesito consultar el saldo de mi tarjeta de crédito.\n\nAgente: Buenas ..."
3,TRS-H4DZT88K2QTZJ4NY5ILY,"Cliente: Buenas tardes, necesito consultar el saldo de mi tarjeta de crédito.\n\nAgente: Buenas ..."
4,TRS-0P221BIAHJQVK8HE3NCE,"Cliente: Buenas tardes, necesito consultar el saldo de mi tarjeta de crédito.\n\nAgente: Buenas ..."


## 12. `mentioned_entities` -- confirm it's still consistently JSON-shaped

In [17]:
print("Sample values:")
display(q_full("SELECT mentioned_entities FROM bronze.call_transcripts WHERE mentioned_entities IS NOT NULL LIMIT 10"))

print()
print("How many populated values fail DuckDB's json_valid() check?")
display(q("""
    SELECT
        count(*) AS populated,
        count(*) FILTER (WHERE NOT json_valid(mentioned_entities)) AS not_valid_json
    FROM bronze.call_transcripts
    WHERE mentioned_entities IS NOT NULL
"""))

Sample values:


,mentioned_entities
0,"{""account_numbers"": 0, ""dates"": 0, ""amounts"": 0, ""products"": ""Cuenta Ahorro""}"
1,"{""account_numbers"": 0, ""dates"": 0, ""amounts"": 1, ""products"": ""Tarjeta Crédito""}"
2,"{""account_numbers"": 0, ""dates"": 1, ""amounts"": 2, ""products"": null}"
3,"{""account_numbers"": 0, ""dates"": 1, ""amounts"": 0, ""products"": null}"
4,"{""account_numbers"": 0, ""dates"": 1, ""amounts"": 0, ""products"": null}"
5,"{""account_numbers"": 2, ""dates"": 0, ""amounts"": 2, ""products"": null}"
6,"{""account_numbers"": 0, ""dates"": 0, ""amounts"": 2, ""products"": null}"
7,"{""account_numbers"": 0, ""dates"": 0, ""amounts"": 0, ""products"": ""Tarjeta Crédito""}"
8,"{""account_numbers"": 0, ""dates"": 1, ""amounts"": 1, ""products"": null}"
9,"{""account_numbers"": 2, ""dates"": 0, ""amounts"": 1, ""products"": ""Tarjeta Crédito""}"



How many populated values fail DuckDB's json_valid() check?


,populated,not_valid_json
0,154157,0


## 13. `detected_keywords`/`detected_intents`/`main_topics` -- free text spot checks

In [18]:
for col in ["detected_keywords", "detected_intents", "main_topics"]:
    print(f"{col} sample:")
    display(q_full(f"SELECT {col} FROM bronze.call_transcripts WHERE {col} IS NOT NULL LIMIT 5"))

detected_keywords sample:


,detected_keywords
0,"banco, cuenta, servicio"
1,"cuenta, banco"
2,"cuenta, banco"
3,"banco, cuenta, servicio"
4,"banco, servicio"


detected_intents sample:


,detected_intents
0,consulta_general
1,consulta_general
2,consulta_general
3,consulta_general
4,consulta_general


main_topics sample:


,main_topics
0,Comercial
1,Queja
2,Queja
3,Técnico
4,Producto


## 14. `process_date` -- consistency against the parent interaction's own `process_date`, window check

In [19]:
q("""
    SELECT
        count(*) AS total,
        count(*) FILTER (WHERE t.process_date = i.process_date) AS matches,
        count(*) FILTER (WHERE t.process_date <> i.process_date) AS differs
    FROM silver.fact_call_transcripts t
    JOIN silver.fact_call_center_interactions i ON i.interaction_id = t.interaction_id
""")

,total,matches,differs
0,171321,171321,0


In [20]:
q("""
    SELECT
        count(*) FILTER (WHERE process_date < DATE '2023-06-17') AS before_window,
        count(*) FILTER (WHERE process_date > DATE '2026-06-17') AS after_window,
        min(process_date) AS earliest, max(process_date) AS latest
    FROM silver.fact_call_transcripts
""")

,before_window,after_window,earliest,latest
0,0,0,2023-06-17,2026-06-17


## Checklist

- [ ] Column inventory (1): all 18 Silver columns present, types correct
- [ ] Row reconciliation (2): 200,000 rows as the dictionary declares, dedup clean
- [ ] NOT NULL audit (3): all 9 declared columns populated -- note `transcription_model`/`duration_seconds` fix candidates
- [ ] Nullable columns (4): population rates make sense
- [ ] Domain checks (5): `detected_language`/`transcription_model`/`audio_quality` real values
- [ ] FK integrity (6): all three FKs reproduced directly
- [ ] Cross-table consistency (7): transcript `customer_id`/`agent_id` matches the parent interaction's
- [ ] `accent_confidence` (8): within 0-1
- [ ] `duration_seconds` (9): no negatives, and does it actually agree with the parent interaction's own duration
- [ ] `detected_accent` (10): domain sanity
- [ ] Template placeholders (11): confirms the known `{monto}`/`{moneda}`/`{limite}` issue and its real scope
- [ ] `mentioned_entities` (12): still consistently JSON-shaped
- [ ] Free text fields (13): population/format sanity
- [ ] `process_date` (14): matches the parent interaction, window check

In [21]:
con.close()
print("closed")

closed
